In [ ]:
"""
Author: Sophie A. Liu
Purpose: isolating local expression activity around each immunofluorescent labeled cell
"""

In [1]:
# importing necessary libraries
import numpy as np
import os
import pandas as pd

In [2]:
# working directory
#os.chdir("path/to/your/working/directory")
os.chdir("I:/Hu Lab/Sophie/1. Cell death/all final data")

In [ ]:
# from block 1.2 NMFs
#V = pd.read_csv("NMF_isoC.csv")
V = pd.read_csv("NMF_Apd1.csv")          # (V)isium data

#S = pd.read_csv("iso_coords.csv")       # (S)pots from IF in microns
S = pd.read_csv("apd1_coords.csv")

In [ ]:
# setting parameters and extracting spatial data. 
v_coords = V[["x", "y"]].to_numpy() / 1.5454      # scaling factor from Loupe browser for pixels to µm
s_coords = S[["x", "y"]].to_numpy()               # already in µm

factor_cols = [c for c in V.columns if c.startswith("factor")]

radius = 40                                       # can easily alter

In [9]:
from scipy.spatial import cKDTree

In [10]:
def inputs(S, V, factor_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # replacing our four imaging channel strings as integers for faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # pulling weights of factors for each bin
    V_factors = V[factor_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_factors

In [ ]:
# counts of each cell type in the neighborhood of a IF-labeled cell treated as a center, as well as some derived metrics.
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # if there's nothing in the radius, then set 0.
    else:
        types = S_cells[idx]
        counts = np.bincount(types, minlength=4)               # = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts         # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    n_tumor = n_alive + n_dying
    total = n_tumor + n_immune + n_endothelial
    exist_dying = 1 if n_dying > 0 else 0

    return counts, n_tumor, total, exist_dying

In [ ]:
# mean factor expression in neighborhood, a better representation than sum bc density effects.
# possible future direction is gaussian decay from center
def get_factor_means(center, v_tree, V_factors, radius):
    idx = v_tree.query_ball_point(center, r=radius)           #  same IF index & neighborhood centers

    if len(idx) == 0:
        return np.zeros(V_factors.shape[1])

    return V_factors[idx].mean(axis=0)

In [27]:
def append_row(center, s_tree, v_tree, S_types, V_factors, radius):

    counts, n_tumor, total, exist_dying = counts_in_radius(
        center, s_tree, S_types, radius
    )

    factor_means = get_factor_means(
        center, v_tree, V_factors, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([n_tumor, total, exist_dying]),
        factor_means
    ])

    return row

In [50]:
# to call for final assembly. if analyzing more factors, vectorize. 
def compute_neighborhoods(
    S, V, s_coords, v_coords, factor_cols, radius):

    s_tree, v_tree, S_types, V_factors = inputs(
        S, V, factor_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_factors = V_factors.shape[1]

    results = np.zeros((n_centers, 9 + n_factors))          # 9 number of metrics and coordinates not including factors

    for i, center in enumerate(s_coords):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_factors, radius
        )

    columns = (
        ["x", "y",
         "n_alive", "n_dying", "n_immune", "n_endothelial",
         "n_tumor", "total", "exist_dying"]
        + list(factor_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [59]:
# running 
df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    factor_cols=factor_cols,
    radius=radius
)

df = df.join(S[["cell_type", "sample"]])            # reattaching metadata

In [60]:
df_clean = df[df["n_tumor"] > 0]                    # use n_endothelial > 0 for vasculature alignment validation

In [53]:
# helps counter dependence by sampling non-overlapping neighborhoods using a greedy algorithm.
# random hard-core thinning. could test Matern soft alternative
def non_overlapping(df, n, radius, seed):
    coords = df[['x', 'y']].to_numpy()
    remaining_idx = np.arange(len(coords))
    rng = np.random.default_rng(seed)

    selected_idx = []

    while len(selected_idx) < n and len(remaining_idx) > 0:
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        tree = cKDTree(coords[remaining_idx])

        # getting rid of all other points in that radius
        neighbors = tree.query_ball_point(coords[pick_i], r=radius)
        to_remove = set(remaining_idx[neighbors])

        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [61]:
# running
df_sub = non_overlapping(df_clean, n = 1000,            # comparability, sampling same number control and treated
                             radius=radius*2,           # being more conservative so 2x
                             seed = 42)

In [63]:
# writing/saving files
df_sub.to_csv("NMF_pd140.csv", index=False)
#df_sub.to_csv("NMF_iso40.csv", index=False)            # repeat above for control condition